In [16]:


# 1) Lee con encoding fallback
last_err = None
for enc in ("utf-8-sig", "cp1252", "latin1"):
    try:
        df = pd.read_csv(fp, sep=";", encoding=enc)
        used_enc = enc
        break
    except Exception as e:
        last_err = e
else:
    raise last_err

print("✅ Leído OK con encoding:", used_enc)

# 2) Limpia nombres de columnas (BOM / espacios)
df.columns = (
    df.columns.astype(str)
      .str.replace("\ufeff", "", regex=False)
      .str.strip()
)

print("Columnas detectadas:", list(df.columns))

# 3) Comprueba que existen las esperadas
needed = {"Islas", "Tipo de dato", "Periodo", "Total"}
missing = needed - set(df.columns)
if missing:
    raise KeyError(f"Faltan columnas esperadas: {missing}. Columnas reales: {list(df.columns)}")

# 4) Value counts + crosstab (sin display)
print("\nTop Islas:\n", df["Islas"].value_counts().head(15))
print("\nTipos de dato:\n", df["Tipo de dato"].value_counts())

ct = pd.crosstab(df["Islas"], df["Tipo de dato"])
print("\nCrosstab Islas x Tipo de dato (head):\n", ct.head(15))

✅ Leído OK con encoding: utf-8-sig
Columnas detectadas: ['Provincias', 'Islas', 'Tipo de dato', 'Periodo', 'Total']

Top Islas:
 Islas
Ibiza y Formentera    7998
Mallorca              7998
Menorca               7998
Fuerteventura         7998
Gran Canaria          7998
Lanzarote             7998
Gomera, La            7998
Hierro, El            7998
Palma, La             7998
Tenerife              7998
Name: count, dtype: int64

Tipos de dato:
 Tipo de dato
Dato base                                            34658
Acumulado en lo que va de año                        34658
Variación anual del acumulado en lo que va de año    34658
Name: count, dtype: int64

Crosstab Islas x Tipo de dato (head):
 Tipo de dato        Acumulado en lo que va de año  Dato base  \
Islas                                                          
Fuerteventura                                2666       2666   
Gomera, La                                   2666       2666   
Gran Canaria                            

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent  # sube de /notebooks a la raíz del repo
fp = PROJECT_ROOT / "data" / "interim" / "air_q" / "daily_tfe.csv"

print("Exists:", fp.exists(), fp)
df = pd.read_csv(fp, parse_dates=["date"])
print(df.shape)
print(df["date"].min(), df["date"].max())
print(df["date"].duplicated().sum())
print(df["date"].nunique(), len(df))

Exists: True C:\dev\projects\heat_mortality_analysis\data\interim\air_q\daily_tfe.csv
(3288, 8)
2016-01-01 00:00:00 2024-12-31 00:00:00
0
3288 3288


In [2]:
expected = pd.date_range("2016-01-01", "2024-12-31", freq="D")
missing = expected.difference(df["date"])
extra = df["date"][~df["date"].isin(expected)]
print("missing days:", len(missing))
print("extra days:", len(extra))

missing days: 0
extra days: 0


In [3]:
print(df["station"].value_counts(dropna=False).head(20))

station
Casa Cuna                          2332
Casa cuna                           729
Vuelta Los Pájaros-Sta Cruz TF      189
Depósito de Tristán-Sta Cruz TF      21
Tome Cano                            14
García Escámez-Sta Cruz TF            2
La Hidalgo-Arafo                      1
Name: count, dtype: int64


In [4]:
for col in ["PM10", "PM2.5", "SO2", "NO2", "O3"]:
    print("\n", col)
    print(df[col].describe())
    print("negativos:", (df[col] < 0).sum())


 PM10
count    3288.000000
mean       52.317149
std        87.922820
min         1.000000
25%        20.750000
50%        32.000000
75%        50.600000
max      1794.000000
Name: PM10, dtype: float64
negativos: 0

 PM2.5
count    3276.000000
mean       15.946215
std        20.668741
min         0.000000
25%         8.000000
50%        12.000000
75%        17.000000
max       407.000000
Name: PM2.5, dtype: float64
negativos: 0

 SO2
count    3258.000000
mean        7.685083
std         7.838501
min         1.000000
25%         3.000000
50%         5.000000
75%        10.000000
max       133.000000
Name: SO2, dtype: float64
negativos: 0

 NO2
count    3238.000000
mean       48.347437
std        29.677549
min         0.000000
25%        25.000000
50%        45.000000
75%        68.000000
max       343.000000
Name: NO2, dtype: float64
negativos: 0

 O3
count    3266.000000
mean       68.363442
std        18.749478
min         1.000000
25%        56.000000
50%        68.000000
75%        

In [5]:
mask = df["PM2.5"].notna() & df["PM10"].notna()
print("PM2.5 > PM10:", (df.loc[mask, "PM2.5"] > df.loc[mask, "PM10"]).sum())

PM2.5 > PM10: 16


In [6]:
df = df.sort_values("date")
print("station changes:", (df["station"] != df["station"].shift()).sum())

station changes: 38


In [8]:
# 1) unificar nombres obvios de estación
df["station"] = df["station"].replace({
    "Casa Cuna": "Casa cuna"
})

# 2) inspeccionar casos PM2.5 > PM10
weird = df[(df["PM2.5"].notna()) & (df["PM10"].notna()) & (df["PM2.5"] > df["PM10"])].copy()
print(weird[["date", "station", "PM10", "PM2.5", "SO2", "NO2", "O3"]].head(20))
print("count:", len(weird))

           date                         station  PM10  PM2.5   SO2   NO2  \
726  2017-12-27  Vuelta Los Pájaros-Sta Cruz TF   8.0    9.0  20.0  89.0   
728  2017-12-29  Vuelta Los Pájaros-Sta Cruz TF   9.0  121.0  14.0  69.0   
736  2018-01-06  Vuelta Los Pájaros-Sta Cruz TF  11.0   14.0   8.0  19.0   
739  2018-01-09  Vuelta Los Pájaros-Sta Cruz TF   5.0    9.0   6.0  69.0   
760  2018-01-30  Vuelta Los Pájaros-Sta Cruz TF  18.0   30.0  12.0  23.0   
762  2018-02-01  Vuelta Los Pájaros-Sta Cruz TF  51.0  191.0  23.0  49.0   
846  2018-04-26  Vuelta Los Pájaros-Sta Cruz TF   5.0    6.0   NaN  10.0   
847  2018-04-27  Vuelta Los Pájaros-Sta Cruz TF   4.0    8.0   NaN  12.0   
848  2018-04-28  Vuelta Los Pájaros-Sta Cruz TF   4.0   12.0   NaN  17.0   
849  2018-04-29  Vuelta Los Pájaros-Sta Cruz TF   4.0    6.0   NaN  13.0   
983  2018-09-10  Vuelta Los Pájaros-Sta Cruz TF  25.0   85.0  16.0  25.0   
1029 2018-10-26  Vuelta Los Pájaros-Sta Cruz TF  37.0   38.0   2.0  68.0   
1039 2018-11

# Para anotar -> 16 días con PM2.5 > PM10; 13 en Vuelta Los Pájaros, 3 en Casa Cuna.

In [9]:
df.to_csv(fp, index=False)
print("Rewritten:", fp)
print(df["station"].value_counts().head(10))

Rewritten: C:\dev\projects\heat_mortality_analysis\data\interim\air_q\daily_tfe.csv
station
Casa cuna                          3061
Vuelta Los Pájaros-Sta Cruz TF      189
Depósito de Tristán-Sta Cruz TF      21
Tome Cano                            14
García Escámez-Sta Cruz TF            2
La Hidalgo-Arafo                      1
Name: count, dtype: int64


## Weekly sanity check

In [13]:
import pandas as pd
fp = PROJECT_ROOT / "data" / "processed" /"tenerife"/ "air_quality" / "weekly_tfe_2016_2024.parquet"
print("Exists:", fp.exists(), fp)

df = pd.read_parquet(fp)

print(df[df["NO2"].isna() | df["O3"].isna()])

Exists: True C:\dev\projects\heat_mortality_analysis\data\processed\tenerife\air_quality\weekly_tfe_2016_2024.parquet
    week_start  year       PM10      PM2.5  SO2  NO2  O3  days_with_pm10  \
239 2020-07-27  2020  43.142857  10.285714  2.0  NaN NaN               7   

     days_missing_pm10  
239                  0  


# Nuevo ingest de visibility


In [6]:
import pandas as pd
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
# fp = Path("data/interim/tenerife/visibility/step4_weekly/visibility_weekly_tfe_2016_2024.parquet")
fp = PROJECT_ROOT / "data" / "interim" /"tenerife"/ "visibility" / "step4_weekly" / "visibility_weekly_tfe_2016_2024.parquet"
print("Exists:", fp.exists(), fp)
df = pd.read_parquet(fp)

print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.head(3))
print(df.tail(3))
print(df["week_start"].min(), df["week_start"].max())
print("duplicated week_start:", df.duplicated(subset=["island", "week_start"]).sum())
print(df.isna().sum().sort_values(ascending=False))

Exists: True C:\dev\projects\heat_mortality_analysis\data\interim\tenerife\visibility\step4_weekly\visibility_weekly_tfe_2016_2024.parquet
(471, 13)
['island', 'week_start', 'low_vis_confirmed_days_week', 'low_vis_confirmed_any_week', 'low_vis_possible_days_week', 'low_vis_possible_any_week', 'low_vis_any_days_week', 'low_vis_any_week', 'confirmed_airports_max_week', 'possible_airports_max_week', 'airports_obs_max_week', 'vis_min_m_week', 'rh_min_pct_week']
island                                      object
week_start                     datetime64[ns, UTC]
low_vis_confirmed_days_week                  int64
low_vis_confirmed_any_week                   int64
low_vis_possible_days_week                   int64
low_vis_possible_any_week                    int64
low_vis_any_days_week                        int64
low_vis_any_week                             int64
confirmed_airports_max_week                  int64
possible_airports_max_week                   int64
airports_obs_max_week       

In [4]:
# 1) una fila por semana
assert df.duplicated(subset=["island", "week_start"]).sum() == 0

# 2) week_start en lunes
ws = pd.to_datetime(df["week_start"])
print("all monday:", (ws.dt.weekday == 0).all())

# 3) orden temporal
print("sorted:", ws.is_monotonic_increasing)

# 4) flags semanales coherentes
print((df["low_vis_confirmed_any_week"] <= df["low_vis_any_week"]).all())
print((df["low_vis_possible_any_week"] <= df["low_vis_any_week"]).all())

# 5) días semanales coherentes
print((df["low_vis_confirmed_days_week"] <= 7).all())
print((df["low_vis_possible_days_week"] <= 7).all())
print((df["low_vis_any_days_week"] <= 7).all())

# 6) any_days al menos tan grande como confirmed/possible
print((df["low_vis_any_days_week"] >= df["low_vis_confirmed_days_week"]).all())
print((df["low_vis_any_days_week"] >= df["low_vis_possible_days_week"]).all())

all monday: True
sorted: True
True
True
True
True
True
True
True


In [5]:
print(df[
    [
        "low_vis_confirmed_days_week",
        "low_vis_possible_days_week",
        "low_vis_any_days_week",
        "confirmed_airports_max_week",
        "possible_airports_max_week",
        "airports_obs_max_week",
        "vis_min_m_week",
        "rh_min_pct_week",
    ]
].describe())

       low_vis_confirmed_days_week  low_vis_possible_days_week  \
count                   471.000000                  471.000000   
mean                      0.087049                    0.263270   
std                       0.296902                    0.719566   
min                       0.000000                    0.000000   
25%                       0.000000                    0.000000   
50%                       0.000000                    0.000000   
75%                       0.000000                    0.000000   
max                       2.000000                    5.000000   

       low_vis_any_days_week  confirmed_airports_max_week  \
count             471.000000                   471.000000   
mean                0.350318                     0.082803   
std                 0.773494                     0.275877   
min                 0.000000                     0.000000   
25%                 0.000000                     0.000000   
50%                 0.000000           

In [24]:
import pandas as pd
from pathlib import Path

#fp = Path("data/interim/tenerife/visibility/step2_daily/isd_daily_tenerife_2016-01-01_2024-12-31.parquet")
fp = PROJECT_ROOT / "data" / "interim" /"tenerife"/ "visibility" / "step2_daily" / "isd_daily_tenerife_2016-01-01_2024-12-31.parquet"
df = pd.read_parquet(fp)
print(df["vis_m"].notna().sum())
print(df["vis_m"].isna().mean())
print(df.groupby(df["date"].dt.year)["vis_m"].apply(lambda s: s.notna().mean()))

print("="*50)
print(df["vis_m"].value_counts(dropna=False).head(30))

print(
    df[df["vis_m"].isin([0, 9, 9999, 10000])]
    [["date", "dt_utc", "vis_m", "vis_qc", "rh_pct", "abs_min_from_target"]]
    .head(50)
)

2798
0.14460409660654233
date
2016    0.926230
2017    0.928767
2018    0.852055
2019    0.846575
2020    0.786885
2021    0.842541
2022    0.838356
2023    0.835616
2024    0.840909
Name: vis_m, dtype: float64
vis_m
30000.0    849
40000.0    504
NaN        473
35000.0    339
25000.0    337
50000.0    192
20000.0    128
45000.0     98
15000.0     59
60000.0     53
55000.0     40
9000.0      19
12000.0     18
10000.0     17
18000.0     16
9999.0      14
65000.0     14
28000.0     11
22000.0      9
70000.0      8
7000.0       7
27000.0      7
70001.0      7
8000.0       7
11000.0      5
6000.0       4
9900.0       4
13000.0      4
19000.0      4
23000.0      4
Name: count, dtype: int64
                          date                    dt_utc    vis_m  vis_qc  \
48   2016-02-18 00:00:00+00:00 2016-02-18 12:00:00+00:00   9999.0     1.0   
62   2016-03-03 00:00:00+00:00 2016-03-03 12:00:00+00:00  10000.0     1.0   
176  2016-06-25 00:00:00+00:00 2016-06-25 12:00:00+00:00  10000.0     1.0   

In [25]:


# fp4 = Path("data/interim/tenerife/visibility/step4_weekly/visibility_weekly_tfe_2016_2024.parquet")
fp4 = PROJECT_ROOT / "data" / "interim" /"tenerife"/ "visibility" / "step4_weekly" / "visibility_weekly_tfe_2016_2024.parquet"

w = pd.read_parquet(fp4)

print(w["vis_min_m_week"].value_counts().head(30))
print(w["vis_min_m_week"].describe())
print(w.isna().sum())

vis_min_m_week
30000.0    137
25000.0    110
20000.0     55
15000.0     36
35000.0     20
9000.0      14
9999.0      13
40000.0     12
12000.0     11
10000.0     10
8000.0       7
18000.0      7
6000.0       4
22000.0      4
11000.0      4
5000.0       3
7000.0       3
9900.0       3
3000.0       2
3500.0       2
17000.0      2
24000.0      2
27000.0      1
2000.0       1
0.0          1
13000.0      1
29000.0      1
3600.0       1
500.0        1
14000.0      1
Name: count, dtype: int64
count      471.000000
mean     22991.055202
std       8534.316252
min          0.000000
25%      16000.000000
50%      25000.000000
75%      30000.000000
max      45000.000000
Name: vis_min_m_week, dtype: float64
island                         0
week_start                     0
low_vis_confirmed_days_week    0
low_vis_confirmed_any_week     0
low_vis_possible_days_week     0
low_vis_possible_any_week      0
low_vis_any_days_week          0
low_vis_any_week               0
confirmed_airports_max_week    0

In [20]:
df.columns.to_list()

['island',
 'week_start',
 'low_vis_confirmed_days_week',
 'low_vis_confirmed_any_week',
 'low_vis_possible_days_week',
 'low_vis_possible_any_week',
 'low_vis_any_days_week',
 'low_vis_any_week',
 'confirmed_airports_max_week',
 'possible_airports_max_week',
 'airports_obs_max_week',
 'vis_min_m_week',
 'rh_min_pct_week']

In [21]:
df["vis_min_m_week"].value_counts().head(30)

vis_min_m_week
9.0        346
25000.0     36
30000.0     32
20000.0     13
15000.0      8
9000.0       6
9999.0       6
12000.0      3
6000.0       2
10000.0      2
17000.0      2
40000.0      2
35000.0      2
8000.0       2
7000.0       1
18000.0      1
24000.0      1
11000.0      1
5000.0       1
0.0          1
9900.0       1
3600.0       1
500.0        1
Name: count, dtype: int64

In [22]:
df[df["vis_min_m_week"].isin([0, 9, 9999, 10000])][
    ["week_start", "vis_min_m_week", "low_vis_any_week", "low_vis_confirmed_days_week"]
].head(30)

,week_start,vis_min_m_week,low_vis_any_week,low_vis_confirmed_days_week
2,2016-01-11 00:00:00+00:00,9.0,1,1
7,2016-02-15 00:00:00+00:00,9999.0,0,0
10,2016-03-07 00:00:00+00:00,9.0,1,1
13,2016-03-28 00:00:00+00:00,9.0,1,2
14,2016-04-04 00:00:00+00:00,9.0,1,2
15,2016-04-11 00:00:00+00:00,9.0,1,1
19,2016-05-09 00:00:00+00:00,9.0,1,1
20,2016-05-16 00:00:00+00:00,9.0,1,1
21,2016-05-23 00:00:00+00:00,9.0,1,1
22,2016-05-30 00:00:00+00:00,9.0,1,1


# revisar el significado de 0.0 y quizá también 70001.0 si sigue apareciendo en Step2

decidir si 9999 y 10000 se tratan como normales o capped, pero eso ya no rompe la ingest

In [26]:
import pandas as pd
df = pd.read_parquet("../data/processed/tenerife/master/master_tfe_2015_2024.parquet")

print(df.loc[df["cap_heat_yellow_plus_week"].isna(), ["week_start"]].head())
print(df.loc[df["cap_heat_yellow_plus_week"].isna(), ["week_start"]].tail())

  week_start
0 2015-12-28
1 2016-01-04
2 2016-01-11
3 2016-01-18
4 2016-01-25
    week_start
124 2018-05-14
125 2018-05-21
126 2018-05-28
127 2018-06-04
128 2018-06-11


## ==== PARA CADA DATASET GENERADO POR CADA ISLA ========

In [1]:
print(df.shape)
print(df.dtypes)
print(df["week_start"].head())
print(df["week_start"].duplicated().sum())
print(df.isna().sum())
print(df.sort_values("week_start").head())
print(df.sort_values("week_start").tail())

assert "week_start" in df.columns
assert df["week_start"].dtype.kind == "M"
assert df["week_start"].duplicated().sum() == 0

df.describe()

bad = df[df["tmax_mean_week"] < df["tmin_mean_week"]]
print(bad[["week_start", "tmin_mean_week", "tmax_mean_week"]].head())

NameError: name 'df' is not defined